# Total available event counts
Counts **all available history** from `/v1/drives` and `/v1/warnings`, without the profiling notebook's `received_since` filter. The API has no documented count endpoint or total-count field, so counting requires traversing every page. Existing filtered pages cannot provide the all-history total.

Counts API rows separately from distinct session/warning IDs. No raw events are saved here; local checkpoints contain counts, IDs, and pagination state under ignored `data/event_counts/`. Completed scans are reused, interrupted scans resume. Delete that directory only when intentionally requesting a fresh count.

This live API provides no documented snapshot guarantee: totals describe this traversal, not a transactionally consistent snapshot. A fixed receipt cutoff also gives the number of observed rows received by the scan's start time; it cannot guarantee absence of later backfills or updates. Safety cap: 2,000 pages per endpoint; an unfinished count is explicitly a lower bound.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")
BASE_URL = "https://qw7ytrkpd3swtndax6c4a73oua0mcrky.lambda-url.eu-west-1.on.aws"
CACHE = Path("data/event_counts")
CACHE.mkdir(parents=True, exist_ok=True)
PAGE_SIZE, MAX_PAGES = 1000, 2000
scope_path = CACHE / "scope.json"
if not scope_path.exists():
    scope_path.write_text(json.dumps({"receipt_cutoff_utc": datetime.now(timezone.utc).isoformat()}))
cutoff = datetime.fromisoformat(json.loads(scope_path.read_text())["receipt_cutoff_utc"])
print("Receipt cutoff (UTC):", cutoff.isoformat())

In [ ]:
session = requests.Session()
authenticated = False

def authenticate():
    global authenticated
    credentials = {"grant_type": "client_credentials", "client_id": os.getenv("OOONO_CLIENT_ID"), "client_secret": os.getenv("OOONO_CLIENT_SECRET")}
    if not credentials["client_id"] or not credentials["client_secret"]:
        raise RuntimeError("Configure local .env credentials.")
    response = session.post(f"{BASE_URL}/oauth/token", data=credentials, timeout=(10, 60))
    if response.status_code != 200:
        raise RuntimeError(f"Authentication HTTP {response.status_code}; body hidden.")
    session.headers.update({"Authorization": "Bearer " + response.json()["access_token"]})
    authenticated = True

def get_page(endpoint, params):
    if not authenticated:
        authenticate()
    for attempt in range(4):
        try:
            response = session.get(f"{BASE_URL}/v1/{endpoint}", params=params, timeout=(10, 60))
        except (requests.Timeout, requests.ConnectionError):
            if attempt == 3:
                raise RuntimeError(f"{endpoint}: request failed; cached pages retained.") from None
            time.sleep(2 ** attempt)
            continue
        if response.status_code == 401 and attempt < 3:
            authenticate()
            continue
        if response.status_code in [429, 500, 502, 503, 504] and attempt < 3:
            wait = response.headers.get("Retry-After", "")
            time.sleep(float(wait) if wait.isdigit() else 2 ** attempt)
            continue
        if response.status_code != 200:
            raise RuntimeError(f"{endpoint}: HTTP {response.status_code}; body hidden.")
        return response.json()
    raise RuntimeError("Retry budget exhausted.")



In [ ]:
def count_endpoint(endpoint, id_field):
    path = CACHE / f"{endpoint}.json"
    if path.exists():
        state = json.loads(path.read_text())
    else:
        state = dict(pages=0, rows=0, rows_by_cutoff=0, missing_ids=0, invalid_receipts=0,
                     ids=[], ids_by_cutoff=[], next_cursor=None, seen_cursors=[], complete=False,
                     receipt_min=None, receipt_max=None, page_sizes=[], response_keys=[])
    ids, cutoff_ids = set(state["ids"]), set(state["ids_by_cutoff"])
    new_calls = 0
    while not state["complete"] and state["pages"] < MAX_PAGES:
        params = {"limit": PAGE_SIZE}
        if state["next_cursor"]:
            params["cursor"] = state["next_cursor"]
        payload = get_page(endpoint, params)
        if not isinstance(payload.get("data"), list) or "next_cursor" not in payload:
            raise RuntimeError("Unexpected page structure; stopping without displaying payload.")
        state["response_keys"] = list(payload)
        cursor = payload["next_cursor"]
        if cursor and cursor in state["seen_cursors"]:
            raise RuntimeError("Repeated cursor; count stopped to avoid a loop.")
        records = payload["data"]
        for record in records:
            event_id = record.get(id_field)
            if event_id is None:
                state["missing_ids"] += 1
            else:
                ids.add(event_id)
            try:
                receipt = datetime.fromisoformat(record["received_at"].replace("Z", "+00:00"))
                if receipt.tzinfo is None:
                    raise ValueError("Receipt has no timezone")
            except (KeyError, TypeError, ValueError):
                state["invalid_receipts"] += 1
                continue
            stamp = receipt.astimezone(timezone.utc).isoformat()
            state["receipt_min"] = min(state["receipt_min"] or stamp, stamp)
            state["receipt_max"] = max(state["receipt_max"] or stamp, stamp)
            if receipt <= cutoff:
                state["rows_by_cutoff"] += 1
                if event_id is not None:
                    cutoff_ids.add(event_id)
        state["rows"] += len(records)
        state["pages"] += 1
        state["next_cursor"] = cursor
        state["complete"] = not bool(cursor)
        state["page_sizes"] = sorted(set(state["page_sizes"] + [len(records)]))
        if cursor:
            state["seen_cursors"].append(cursor)
        state["ids"], state["ids_by_cutoff"] = sorted(ids), sorted(cutoff_ids)
        state["last_saved_utc"] = datetime.now(timezone.utc).isoformat()
        temp = path.with_suffix(".tmp")
        temp.write_text(json.dumps(state))
        temp.replace(path)
        new_calls += 1
        if new_calls % 50 == 0:
            print(f"{endpoint}: {state['pages']} pages, {state['rows']:,} rows counted", flush=True)
    return {"endpoint": endpoint, "pagination_exhausted": state["complete"], "pages": state["pages"],
            "new_calls": new_calls, "API_rows": state["rows"], "distinct_IDs": len(ids),
            "extra_rows_on_repeated_IDs": state["rows"] - state["missing_ids"] - len(ids),
            "missing_IDs": state["missing_ids"], "rows_received_by_cutoff": state["rows_by_cutoff"],
            "distinct_IDs_by_cutoff": len(cutoff_ids), "invalid_receipts": state["invalid_receipts"],
            "receipt_min_utc": state["receipt_min"], "receipt_max_utc": state["receipt_max"],
            "observed_page_sizes": str(state["page_sizes"]), "scan_last_saved_utc": state["last_saved_utc"]}

results = pd.DataFrame([count_endpoint("drives", "session_id"), count_endpoint("warnings", "warning_id")])
display(results.set_index("endpoint").T)
if results["pagination_exhausted"].all():
    print(f"Total observed API event rows: {results['API_rows'].sum():,}")
    print(f"Total distinct IDs across the two event types: {results['distinct_IDs'].sum():,}")
else:
    print("Pagination cap reached: these are lower bounds, not total counts. Raise MAX_PAGES to resume.")

`API_rows` includes repeated IDs; `distinct_IDs` measures session IDs for drives and warning IDs for warnings. Summing those distinct counts counts two event types, not distinct drives or users. `rows_received_by_cutoff` excludes records received after the fixed cutoff. Receipt filtering cannot remove concurrent historical backfills. Run this notebook again to reuse completed results without API calls. No credentials or cursors are displayed.